# # 使用贝叶斯优化得到的最佳超参数进行模型训练
#
# 本notebook旨在利用之前贝叶斯优化得到的最佳超参数组合，重新进行一次完整的模型训练、验证和测试。
# 我们将尽可能复用项目中的现有代码模块。


In [ ]:
import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR, CosineAnnealingLR, ReduceLROnPlateau
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm # 使用notebook版本的tqdm
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, confusion_matrix

# 导入项目中的模块
# 假设这些模块在当前工作目录或Python路径下
from config import load_config, save_config, CONFIG # 导入CONFIG以获取一些默认值或结构
from data import load_data #
from models import get_model #
from visualization import visualize_training_curves, visualize_confusion_matrix, visualize_dataset_distribution #


In [ ]:
# ## 2. 定义和加载配置
#
# 首先，加载基础配置，然后我们将用贝叶斯优化得到的最佳超参数覆盖它。

# %%
# 加载默认配置
# 如果有特定的配置文件，可以在这里指定路径，例如 'path/to/your_specific_config.json'
# 否则，将使用 config.py 中的默认 CONFIG
cfg = load_config() #

In [ ]:
# ## 3. 定义最佳超参数
#
# 从您的日志中提取最佳试验的参数。
# 日志示例: `Trial 82 finished with value: 0.8431010808190281 and parameters: {'learning_rate': ..., 'optimizer': 'adamw', ...}`
# **请将以下 `best_hyperparams` 字典替换为您日志中 `Best is trial ... with value: ...` 后面跟着的参数字典。**

# %%
# 示例日志: [I 2025-05-17 00:58:12,576] Trial 99 finished with value: 0.837090108280788 and parameters: {'learning_rate': 9.191261837889327e-05, 'weight_decay': 0.0005175833650131985, 'optimizer': 'adamw', 'dropout_rate': 0.19317216698770392, 'activation': 'gelu', 'lr_scheduler': 'step', 'model_type': 'base_mlp', 'layer_sizes_idx': 0, 'step_size': 4, 'step_gamma': 0.1590396939768399}. Best is trial 82 with value: 0.8431010808190281.
# 我们需要 Trial 82 的参数。假设Trial 82的参数如下（您需要从您的实际日志中获取）：
# 例如，如果Trial 82的参数是:
# best_hyperparams_from_log = {
# 'learning_rate': 1e-4,
# 'weight_decay': 1e-5,
# 'optimizer': 'adamw',
# 'dropout_rate': 0.3,
# 'activation': 'relu',
# 'lr_scheduler_type': 'cosine', # 对应日志中的 'lr_scheduler'
# 'model_type': 'base_mlp',
# 'hidden_units': [2048, 2048, 1024], # 假设 'layer_sizes_idx' 对应于某个预定义的hidden_units，或者优化直接输出了hidden_units
# # 如果日志中有 'step_size' 和 'step_gamma' (用于 'step' scheduler)，也加入进来
# # 'lr_milestones': [10, 20], # for multistep
# }


best_hyperparams_from_log = {
    'learning_rate': 9.191261837889327e-05, # 从您提供的 Trial 99 日志中获取，假设这是您想用的
    'weight_decay': 0.0005175833650131985,
    'optimizer': 'adamw',
    'dropout_rate': 0.19317216698770392,
    'activation': 'gelu',
    'lr_scheduler_type': 'step', #  optuna中可能是 'lr_scheduler'
    'model_type': 'base_mlp',
    # 'layer_sizes_idx': 0, # 这个需要映射到实际的 hidden_units
    # 假设 layer_sizes_idx = 0 对应于 config.py 中的默认 hidden_units
    # 或者，如果您的优化过程直接优化了 hidden_units，请在这里提供
    'hidden_units': cfg.get('hidden_units', [4096, 4096, 4096, 4096]), # 使用cfg的默认值或您优化的值
    'lr_step_size': 4, # 对应日志中的 'step_size'
    'lr_gamma': 0.1590396939768399, # 对应日志中的 'step_gamma'
}

# 更新配置
for key, value in best_hyperparams_from_log.items():
    # 特殊处理一些可能名称不一致的参数
    if key == 'learning_rate':
        cfg['lr'] = value
    elif key == 'lr_scheduler': # Optuna中可能叫lr_scheduler
        cfg['lr_scheduler_type'] = value
    elif key == 'step_size': # Optuna中可能叫step_size
        cfg['lr_milestones'] = [value * i for i in range(1, cfg['epochs'] // value +1 )] # 假设step_size用于MultiStepLR或者等效的StepLR
        # 或者如果你的scheduler是StepLR, 那么这个就是 step_size
        cfg['lr_step_size'] = value
    elif key == 'step_gamma':
         cfg['lr_gamma'] = value
    else:
        cfg[key] = value

# 其他可能需要根据 best_hyperparams_from_log 调整的配置
cfg['epochs'] = cfg.get('epochs', 30) # 确保 epochs 参数存在，如果优化了epochs，也应包含在best_hyperparams_from_log中
cfg['batch_size'] = cfg.get('batch_size', 128)
cfg['val_epochs'] = cfg.get('val_epochs', 3)
cfg['use_lr_scheduler'] = True # 既然优化了scheduler参数，就启用它

print("更新后的配置:")
for key, value in cfg.items():
    if key in best_hyperparams_from_log or key in ['lr', 'lr_scheduler_type', 'lr_milestones', 'lr_gamma', 'epochs', 'batch_size', 'val_epochs', 'use_lr_scheduler']:
        print(f"  {key}: {value}")


In [ ]:
# ## 4. 设置设备 (CPU/GPU)

# %%
device = torch.device(f"cuda:{cfg['device']}" if cfg['device'] >= 0 and torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

torch.manual_seed(cfg['random_seed'])
if device.type == 'cuda':
    torch.cuda.manual_seed_all(cfg['random_seed'])
np.random.seed(cfg['random_seed'])


In [ ]:
# ## 5. 加载数据
#
# 使用 `data.load_data` 函数加载训练、验证和测试数据。

# %%
dataset_dict, train_loader, val_loader, test_loader = load_data(cfg, mode='train') #

# 更新配置中的特征维度和类别数 (load_data 内部可能会更新这些)
if 'feature_dim' in dataset_dict:
    cfg['feature_dim'] = dataset_dict['feature_dim']
if 'num_classes' in dataset_dict: # mat_loader.py 的 process_train38_data 会返回 num_classes
    cfg['num_class'] = dataset_dict['num_classes']
elif 'train_labels' in dataset_dict: # 兼容旧版
    # load_multiclass_data 返回的 dataset_dict 没有 num_classes，但有 train_labels
    # labels 应该是 0 到 N-1
    all_labels = np.unique(np.concatenate([
        dataset_dict['train_labels'],
        dataset_dict['test_labels'],
        dataset_dict['val_labels']
    ]))
    # BrainVoxelDataset 和 BrainVoxelMatDataset 都会将标签映射到 0 到 num_classes-1
    # 所以 num_class 应该是唯一标签的数量
    cfg['num_class'] = len(all_labels)


print(f"特征维度 (feature_dim): {cfg['feature_dim']}")
print(f"类别数量 (num_class): {cfg['num_class']}")

# 可视化数据集分布 (可选)
# visualize_dataset_distribution(dataset_dict, save_path=os.path.join(cfg['save_dir'], "dataset_distribution.png"))


In [ ]:
# ## 6. 初始化模型
#
# 使用 `models.get_model` 函数和更新后的配置来创建模型。

# %%
model_params = {
    'input_dim': cfg['feature_dim'],
    'hidden_dims': cfg['hidden_units'],
    'num_classes': cfg['num_class'],
    'dropout_rate': cfg['dropout_rate'],
    'activation': cfg['activation']
}

# 添加模型特有的参数 (如果存在于cfg中)
if cfg['model_type'] == 'deep_mlp':
    model_params['use_skip_connections'] = cfg.get('use_skip_connections', False) #
elif cfg['model_type'] == 'residual_mlp':
    model_params['use_bottleneck'] = cfg.get('use_bottleneck', False) #
    model_params['bottleneck_factor'] = cfg.get('bottleneck_factor', 0.5) #


model = get_model(cfg['model_type'], **model_params) #
model.to(device)

print(f"模型类型: {cfg['model_type']}")
print(f"模型结构: {model}")
# 打印模型参数量
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"总参数量: {total_params:,}")
print(f"可训练参数量: {trainable_params:,}")



In [ ]:

# %%
# Optimizer
if cfg['optimizer'].lower() == 'adam':
    optimizer = optim.Adam(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
elif cfg['optimizer'].lower() == 'adamw':
    optimizer = optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
else:
    raise ValueError(f"不支持的优化器: {cfg['optimizer']}")

# Learning Rate Scheduler
scheduler = None
if cfg['use_lr_scheduler']:
    if cfg['lr_scheduler_type'].lower() == 'multistep':
        scheduler = StepLR(optimizer, step_size=cfg['lr_milestones'][0], gamma=cfg['lr_gamma']) # StepLR is similar to MultiStepLR with regular steps
    elif cfg['lr_scheduler_type'].lower() == 'step': # for step_size and step_gamma from log
        scheduler = StepLR(optimizer, step_size=cfg.get('lr_step_size', 5), gamma=cfg.get('lr_gamma',0.1))
    elif cfg['lr_scheduler_type'].lower() == 'cosine':
        scheduler = CosineAnnealingLR(optimizer, T_max=cfg['epochs'], eta_min=1e-7)
    elif cfg['lr_scheduler_type'].lower() == 'plateau':
        scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=cfg['lr_gamma'], patience=5, verbose=True) #通常基于验证指标
    else:
        print(f"未知的学习率调度器类型: {cfg['lr_scheduler_type']}. 不使用调度器。")

# Loss Function
# config.py 中定义了 ignore_index，但 mat_loader.py 的 _auto_detect_data_format 将其设置为 None
# BrainVoxelDataset 和 BrainVoxelMatDataset 会将标签映射到 0 到 num_classes-1
# 因此，不需要 ignore_index，除非背景类别0仍然存在并且你想忽略它（但代码似乎是过滤掉了背景）
criterion = nn.CrossEntropyLoss()

print(f"优化器: {cfg['optimizer']}")
if scheduler:
    print(f"学习率调度器: {cfg['lr_scheduler_type']}")
print(f"损失函数: CrossEntropyLoss")


In [ ]:
# ## 8. 训练与验证循环

# %%
training_results = {
    'loss_list': [],
    'acc_list': [],
    'val_epoch_list': [],
    'val_acc_list': [],
    'val_f1_macro_list': [],
    'val_kappa_list': []
}

best_val_acc = 0.0
best_model_path = os.path.join(cfg['save_dir'], f"{cfg['model_name']}_best_from_notebook.pth")

print(f"开始训练模型: {cfg['model_name']}")
print(f"总轮数 (Epochs): {cfg['epochs']}")
print(f"验证频率 (Val Epochs): {cfg['val_epochs']}")

for epoch in range(1, cfg['epochs'] + 1):
    model.train()
    epoch_loss = 0
    train_preds = []
    train_targets = []

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{cfg['epochs']} [Train]", leave=False)
    for batch_idx, (data, target) in enumerate(progress_bar):
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        preds = torch.argmax(output, dim=1)
        train_preds.extend(preds.cpu().numpy())
        train_targets.extend(target.cpu().numpy())
        
        if batch_idx % 100 == 0: # 每100个batch打印一次学习率
             progress_bar.set_postfix(loss=loss.item(), lr=optimizer.param_groups[0]['lr'])


    avg_epoch_loss = epoch_loss / len(train_loader)
    epoch_train_acc = accuracy_score(train_targets, train_preds)

    training_results['loss_list'].append(avg_epoch_loss)
    training_results['acc_list'].append(epoch_train_acc)

    print(f"Epoch {epoch}/{cfg['epochs']} - Train Loss: {avg_epoch_loss:.4f}, Train Acc: {epoch_train_acc:.4f}, LR: {optimizer.param_groups[0]['lr']:.2e}")

    if epoch % cfg['val_epochs'] == 0:
        model.eval()
        val_loss = 0
        val_preds = []
        val_targets = []
        with torch.no_grad():
            progress_bar_val = tqdm(val_loader, desc=f"Epoch {epoch}/{cfg['epochs']} [Val]", leave=False)
            for data, target in progress_bar_val:
                data, target = data.to(device), target.to(device)
                output = model(data)
                loss = criterion(output, target)
                val_loss += loss.item()
                preds = torch.argmax(output, dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_targets.extend(target.cpu().numpy())

        avg_val_loss = val_loss / len(val_loader)
        val_acc = accuracy_score(val_targets, val_preds)
        val_f1 = f1_score(val_targets, val_preds, average='macro', zero_division=0)
        val_kappa = cohen_kappa_score(val_targets, val_preds)

        training_results['val_epoch_list'].append(epoch)
        training_results['val_acc_list'].append(val_acc)
        training_results['val_f1_macro_list'].append(val_f1)
        training_results['val_kappa_list'].append(val_kappa)

        print(f"  Validation - Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.4f}, Val F1 (Macro): {val_f1:.4f}, Val Kappa: {val_kappa:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            if cfg['save_checkpoints']:
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'val_acc': val_acc,
                    'config': cfg # 保存当时的配置
                }, best_model_path)
                print(f"   mejor modelo guardado en {best_model_path} con Val Acc: {best_val_acc:.4f}")
        
        if scheduler:
            if cfg['lr_scheduler_type'].lower() == 'plateau':
                scheduler.step(val_acc) # ReduceLROnPlateau needs a metric
            elif cfg['lr_scheduler_type'].lower() != 'plateau' and scheduler is not None:
                 scheduler.step() # For other schedulers like StepLR, CosineAnnealingLR
    elif scheduler and cfg['lr_scheduler_type'].lower() != 'plateau': # Step scheduler at each epoch if not plateau
        scheduler.step()


print("Entrenamiento completado.")


In [ ]:
# ## 9. 可视化训练曲线

# %%
curves_save_path = os.path.join(cfg['save_dir'], f"{cfg['model_name']}_training_curves_from_notebook.png")
visualize_training_curves(training_results, save_path=curves_save_path) #


In [ ]:
# ## 10. 在测试集上评估模型
#
# 加载性能最佳的模型，并在测试集上进行评估。

# %%
if cfg['save_checkpoints'] and os.path.exists(best_model_path):
    print(f"Cargando el mejor modelo desde: {best_model_path}")
    checkpoint = torch.load(best_model_path, map_location=device)
    
    # 从checkpoint中恢复模型参数以重新构建模型 (如果需要的话)
    # saved_cfg = checkpoint.get('config', cfg) # 获取保存的配置
    # model_params_loaded = {
    #     'input_dim': saved_cfg['feature_dim'],
    #     'hidden_dims': saved_cfg['hidden_units'],
    #     'num_classes': saved_cfg['num_class'],
    #     'dropout_rate': saved_cfg['dropout_rate'],
    #     'activation': saved_cfg['activation']
    # }
    # if saved_cfg['model_type'] == 'deep_mlp': model_params_loaded['use_skip_connections'] = saved_cfg.get('use_skip_connections', False)
    # elif saved_cfg['model_type'] == 'residual_mlp':
    #     model_params_loaded['use_bottleneck'] = saved_cfg.get('use_bottleneck', False)
    #     model_params_loaded['bottleneck_factor'] = saved_cfg.get('bottleneck_factor', 0.5)
    # model = get_model(saved_cfg['model_type'], **model_params_loaded)
    
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
else:
    print("No se encontró el checkpoint del mejor modelo, se utilizará el modelo actual.")

model.eval()
test_preds = []
test_targets = []
with torch.no_grad():
    progress_bar_test = tqdm(test_loader, desc="Testing", leave=False)
    for data, target in progress_bar_test:
        data, target = data.to(device), target.to(device)
        output = model(data)
        preds = torch.argmax(output, dim=1)
        test_preds.extend(preds.cpu().numpy())
        test_targets.extend(target.cpu().numpy())

test_acc = accuracy_score(test_targets, test_preds)
test_f1_macro = f1_score(test_targets, test_preds, average='macro', zero_division=0)
test_f1_weighted = f1_score(test_targets, test_preds, average='weighted', zero_division=0)
test_kappa = cohen_kappa_score(test_targets, test_preds)

print("\nResultados en el conjunto de Test:")
print(f"  Accuracy: {test_acc:.4f}")
print(f"  F1 Score (Macro): {test_f1_macro:.4f}")
print(f"  F1 Score (Weighted): {test_f1_weighted:.4f}")
print(f"  Cohen's Kappa: {test_kappa:.4f}")


In [ ]:
# ## 11. 可视化混淆矩阵

# %%
conf_matrix = confusion_matrix(test_targets, test_preds)
cm_save_path = os.path.join(cfg['save_dir'], f"{cfg['model_name']}_confusion_matrix_from_notebook.png")
visualize_confusion_matrix(conf_matrix, save_path=cm_save_path, log_scale=True) #

# %% [markdown]
# ## 12. 保存最终结果和配置 (可选)

# %%
final_results = {
    'best_validation_accuracy': best_val_acc,
    'test_accuracy': test_acc,
    'test_f1_macro': test_f1_macro,
    'test_f1_weighted': test_f1_weighted,
    'test_kappa': test_kappa,
    'training_results': training_results,
    'hyperparameters_used': cfg # 保存实际使用的配置
}

results_save_path = os.path.join(cfg['save_dir'], f"{cfg['model_name']}_final_results_from_notebook.json")
with open(results_save_path, 'w') as f:
    # 对于不可序列化的 torch.device，转换为字符串
    if 'device' in final_results['hyperparameters_used'] and not isinstance(final_results['hyperparameters_used']['device'], str):
        final_results['hyperparameters_used']['device'] = str(final_results['hyperparameters_used']['device'])
    # 移除 scaler 和 pca_model (如果存在且不可序列化)
    if 'scaler' in final_results['hyperparameters_used']:
        del final_results['hyperparameters_used']['scaler']
    if 'pca_model' in final_results['hyperparameters_used']:
        del final_results['hyperparameters_used']['pca_model']

    json.dump(final_results, f, indent=4, default=lambda o: '<not serializable>')


# 保存最终使用的配置 (如果需要)
# final_config_save_path = os.path.join(cfg['save_dir'], f"{cfg['model_name']}_config_from_notebook.json")
# save_config(cfg, final_config_save_path) #

print(f"Resultados finales guardados en: {results_save_path}")
print("Notebook completado.")
